In [1]:
import os
os.environ['CUDA_LAUNCH_BLOCKING'] = '1'

print("✓ CUDA_LAUNCH_BLOCKING set to:", os.environ['CUDA_LAUNCH_BLOCKING'])

%load_ext autoreload
%autoreload 2

✓ CUDA_LAUNCH_BLOCKING set to: 1


In [ ]:
!nvidia-smi

import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
torch.cuda.empty_cache()

print(f"Using device: {DEVICE}")


/bin/bash: line 1: nvidia-smi: command not found


#Load from Google Drive

In [3]:
# from google.colab import drive
# drive.mount('/content/drive')

# # Copy Python files FROM Drive TO Colab
# !cp /content/drive/MyDrive/AST_Model/ast_models.py /content/
# !cp /content/drive/MyDrive/AST_Model/dataset.py /content/
# !cp /content/drive/MyDrive/AST_Model/train.py /content/

# # Unzip data FROM Drive TO Colab
# #!unzip -q /content/drive/MyDrive/AST_Model/data/training.zip -d /content/
# !cp /content/drive/MyDrive/AST_Model/data/file_labels.csv /content/
# !cp /content/drive/MyDrive/AST_Model/preprocess.py /content/
# !cp -r /content/drive/MyDrive/mel_spectrograms /content/
# !cp /content/drive/MyDrive/AST_Model/requirements_colab.txt /content/

# !cp /content/drive/MyDrive/stage1_als_detector.pth /content/
# !cp /content/drive/MyDrive/stage2_dysarthria_detector.pth /content/
# !unzip -q /content/drive/MyDrive/AST_Model/data/mel_spectrograms_test.zip -d /content/

# # Verify everything is in Colab now
# !ls -la /content/


ModuleNotFoundError: No module named 'google'

#Install Requirements

In [1]:
pip install -r requirements_colab.txt

  Using cached timm-0.4.5-py3-none-any.whl.metadata (24 kB)
  Using cached wget-3.2-py3-none-any.whl
  Using cached librosa-0.11.0-py3-none-any.whl.metadata (8.7 kB)
  Using cached soundfile-0.13.1-py2.py3-none-manylinux_2_28_x86_64.whl.metadata (16 kB)
  Using cached scikit_learn-1.7.2-cp312-cp312-manylinux2014_x86_64.manylinux_2_17_x86_64.whl.metadata (11 kB)
  Using cached pandas-2.3.3-cp312-cp312-manylinux_2_24_x86_64.manylinux_2_28_x86_64.whl.metadata (91 kB)
  Using cached audioread-3.1.0-py3-none-any.whl.metadata (9.0 kB)
  Using cached numba-0.62.1-cp312-cp312-manylinux2014_x86_64.manylinux_2_17_x86_64.whl.metadata (2.8 kB)
  Using cached scipy-1.16.3-cp312-cp312-manylinux2014_x86_64.manylinux_2_17_x86_64.whl.metadata (62 kB)
  Using cached joblib-1.5.2-py3-none-any.whl.metadata (5.6 kB)
  Using cached pooch-1.8.2-py3-none-any.whl.metadata (10 kB)
  Using cached soxr-1.0.0-cp312-abi3-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (5.6 kB)
  Using cached lazy_loader-0.

Import Libraries

In [4]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, random_split, Subset
import sys
import os

from dataset import DysarthriaDataset
from ast_models import ASTModel


ModuleNotFoundError: No module named 'wget'

#Create ALS Dataset

In [3]:
# Stage 1: ALS Detection
class ALSDataset(DysarthriaDataset):
    def __init__(self, spectrogram_path, label_file, use_phonation='all'):
        # Call parent class constructor
        super().__init__(spectrogram_path, label_file, use_phonation = use_phonation)

    def __getitem__(self, idx):
        spectrogram, original_label = super().__getitem__(idx)

        # Class 5 (Healthy) → 0
        # Classes 1,2,3,4 (ALS) → 1
        label = 0 if original_label == 4 else 1
        return spectrogram, label

#Create Dysarthria Detection Dataset

In [4]:
# Stage 2: Dysarthria Detection (only on ALS patients)
class DysarthriaDetectionDataset(DysarthriaDataset):
    def __init__(self, spectrogram_path, label_file, use_phonation = 'all'):
        super().__init__(spectrogram_path, label_file, use_phonation = use_phonation)
        # Filter to keep only ALS patients (classes 1,2,3,4)
        self.filtered_indices = [i for i in range(len(self.labels))
                                if self.labels[i] in [1, 2, 3, 4]]

    def __len__(self):
        return len(self.filtered_indices)

    def __getitem__(self, idx):
        real_idx = self.filtered_indices[idx]
        spectrogram, original_label = super().__getitem__(real_idx)
        # Class 4 (ALS without dysarthria) → 0
        # Classes 1,2,3 (ALS with dysarthria) → 1
        label = 0 if original_label == 3 else 1
        return spectrogram, label

#Create Dysarthria Severity Dataset

In [5]:
# Stage 3: Severity Classification (only on dysarthric patients)
class SeverityDataset(DysarthriaDataset):
    def __init__(self, spectrogram_path, label_file, use_phonation = 'all'):
        super().__init__(spectrogram_path, label_file, use_phonation = use_phonation)
        # Filter to keep only dysarthric patients (classes 1,2,3)
        self.filtered_indices = [i for i in range(len(self.labels))
                                if self.labels[i] in [1,2,3]]

    def __len__(self):
        return len(self.filtered_indices)

    def __getitem__(self, idx):
        real_idx = self.filtered_indices[idx]
        spectrogram, original_label = super().__getitem__(real_idx)
        # Map: 1→2 (Severe), 2→1 (Moderate), 3→0 (Mild)
        # Or keep as: 1→0, 2→1, 3→2
        label = original_label  # Shift to 0-indexed
        return spectrogram, label

Model Coefficients

#Stage 1: Train ALS Detection

In [7]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Subset
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, f1_score
import numpy as np
NUM_EPOCHS = 10
BATCH_SIZE = 8
LEARNING_RATE = 1e-4
# ============================================================================
# GENERIC TRAINING FUNCTION (REUSABLE FOR ALL STAGES)
# ============================================================================
def train_binary_classifier(
    dataset,
    stage_name,
    model_save_name,
    class_names,
    label_dim=2,
    batch_size=16,
    num_epochs=5,
    learning_rate=1e-4,
    device=None
):
    """
    Generic training function for binary classification stages
    
    Parameters:
    -----------
    dataset : Dataset
        The dataset to train on
    stage_name : str
        Name of the stage (e.g., "STAGE 1: ALS DETECTION")
    model_save_name : str
        Base name for saving models (e.g., "stage1_als_detector")
    class_names : list
        List of class names [class_0_name, class_1_name]
    label_dim : int
        Number of output classes (default: 2)
    batch_size : int
        Batch size for training
    num_epochs : int
        Number of training epochs
    learning_rate : float
        Learning rate
    device : torch.device
        Device to train on (default: auto-detect)
    
    Returns:
    --------
    model : ASTModel
        Trained model
    test_idx : list
        Test set indices for hierarchical evaluation
    """
    
    if device is None:
        device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    
    print(f"\n{'='*70}")
    print(f"{stage_name} - DATASET SPLITTING")
    print(f"{'='*70}")
    print(f"Total samples: {len(dataset)}")
    
    # ========================================================================
    # PROPER STRATIFIED TRAIN/VAL/TEST SPLIT
    # ========================================================================
    all_labels = [dataset[i][1] for i in range(len(dataset))]
    indices = list(range(len(dataset)))
    
    # First split: Train (70%) vs Temp (30%)
    train_idx, temp_idx = train_test_split(
        indices,
        test_size=0.3,
        stratify=all_labels,
        random_state=42
    )
    
    # Second split: Val (15%) vs Test (15%)
    temp_labels = [all_labels[i] for i in temp_idx]
    val_idx, test_idx = train_test_split(
        temp_idx,
        test_size=0.5,
        stratify=temp_labels,
        random_state=42
    )
    
    # Show split statistics
    train_labels = [all_labels[i] for i in train_idx]
    val_labels = [all_labels[i] for i in val_idx]
    test_labels = [all_labels[i] for i in test_idx]
    
    print(f"\n📊 Split sizes:")
    print(f"  Train: {len(train_idx)} samples ({len(train_idx)/len(dataset)*100:.1f}%)")
    print(f"  Val:   {len(val_idx)} samples ({len(val_idx)/len(dataset)*100:.1f}%)")
    print(f"  Test:  {len(test_idx)} samples ({len(test_idx)/len(dataset)*100:.1f}%)")
    
    print(f"\n📊 Label distribution per split:")
    for label_val in sorted(set(all_labels)):
        train_count = train_labels.count(label_val)
        val_count = val_labels.count(label_val)
        test_count = test_labels.count(label_val)
        label_name = class_names[label_val] if label_val < len(class_names) else f"Class {label_val}"
        print(f"  {label_name} (Label {label_val}):")
        print(f"    Train: {train_count}, Val: {val_count}, Test: {test_count}")
    
    # Create dataset subsets
    train_dataset = Subset(dataset, train_idx)
    val_dataset = Subset(dataset, val_idx)
    test_dataset = Subset(dataset, test_idx)
    
    # Create data loaders
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)
    
    print(f"\n✓ Data loaders created")
    print(f"{'='*70}\n")
    
    # ========================================================================
    # MODEL INITIALIZATION
    # ========================================================================
    model = ASTModel(
        label_dim=label_dim,
        fstride=10,
        tstride=10,
        input_fdim=128,
        input_tdim=1024,
        imagenet_pretrain=True,
        audioset_pretrain=False,
        model_size='base384'
    )
    model = model.to(device)
    
    # Loss and optimizer
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.SGD(
        model.parameters(),
        lr=learning_rate,
        momentum=0.9,
        weight_decay=0.01
    )
    
    scheduler = torch.optim.lr_scheduler.StepLR(
        optimizer,
        step_size=3,
        gamma=0.5
    )
    
    best_val_acc = 0.0
    best_val_f1 = 0.0
    
    # ========================================================================
    # TRAINING LOOP
    # ========================================================================
    print(f"{'='*70}")
    print("TRAINING")
    print(f"{'='*70}\n")
    
    for epoch in range(num_epochs):
        # Training phase
        model.train()
        train_loss = 0
        train_correct = 0
        train_total = 0
        
        for batch_idx, (spectrograms, labels) in enumerate(train_loader):
            spectrograms = spectrograms.to(device)
            labels = labels.to(device)
            
            # Forward pass
            outputs = model(spectrograms)
            loss = criterion(outputs, labels)
            
            # Backward pass
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            
            # Stats
            train_loss += loss.item()
            _, predicted = outputs.max(1)
            train_total += labels.size(0)
            train_correct += predicted.eq(labels).sum().item()
            
            if batch_idx % 10 == 0:
                print(f'Epoch {epoch+1}/{num_epochs}, Batch {batch_idx}, Loss: {loss.item():.4f}')
        
        # Calculate training metrics
        train_loss = train_loss / len(train_loader)
        train_acc = 100. * train_correct / train_total
        
        # Validation phase
        model.eval()
        val_loss = 0
        val_correct = 0
        val_total = 0
        val_preds = []
        val_true = []
        
        with torch.no_grad():
            for spectrograms, labels in val_loader:
                spectrograms = spectrograms.to(device)
                labels = labels.to(device)
                
                outputs = model(spectrograms)
                loss = criterion(outputs, labels)
                
                val_loss += loss.item()
                _, predicted = outputs.max(1)
                val_total += labels.size(0)
                val_correct += predicted.eq(labels).sum().item()
                
                val_preds.extend(predicted.cpu().numpy())
                val_true.extend(labels.cpu().numpy())
        
        # Calculate validation metrics
        val_loss = val_loss / len(val_loader)
        val_acc = 100. * val_correct / val_total
        val_f1 = f1_score(val_true, val_preds, average='binary')
        
        # Step the scheduler
        current_lr = optimizer.param_groups[0]['lr']
        scheduler.step()
        
        # Print epoch summary
        print(f'\n{"="*60}')
        print(f'Epoch {epoch+1}/{num_epochs} Summary:')
        print(f'Learning Rate: {current_lr:.6f}')
        print(f'Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.2f}%')
        print(f'Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.2f}% | Val F1: {val_f1:.4f}')
        print(f'{"="*60}\n')
        
        # Save best model based on F1 score
        if val_f1 > best_val_f1:
            best_val_f1 = val_f1
            best_val_acc = val_acc
            torch.save({
                'epoch': epoch,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'val_acc': val_acc,
                'val_f1': val_f1
            }, f'best_{model_save_name}.pth')
            print(f'💾 Saved new best model (Val F1: {val_f1:.4f})\n')
    
    print("Training complete!")
    
     # ========================================================================
    # FINAL TEST EVALUATION
    # ========================================================================
    print(f"\n{'='*70}")
    print("FINAL TEST SET EVALUATION")
    print(f"{'='*70}\n")
    
    # Load best model
    checkpoint = torch.load(f'best_{model_save_name}.pth')
    model.load_state_dict(checkpoint['model_state_dict'])
    print(f"✓ Loaded best model from epoch {checkpoint['epoch']+1}")
    print(f"  Val Accuracy: {checkpoint['val_acc']:.2f}%")
    print(f"  Val F1: {checkpoint['val_f1']:.4f}\n")
    
    # Evaluate on test set
    model.eval()
    test_loss = 0
    test_correct = 0
    test_total = 0
    test_preds = []
    test_true = []
    
    with torch.no_grad():
        for spectrograms, labels in test_loader:
            spectrograms = spectrograms.to(device)
            labels = labels.to(device)
            
            outputs = model(spectrograms)
            loss = criterion(outputs, labels)
            
            test_loss += loss.item()
            _, predicted = outputs.max(1)
            test_total += labels.size(0)
            test_correct += predicted.eq(labels).sum().item()
            
            test_preds.extend(predicted.cpu().numpy())
            test_true.extend(labels.cpu().numpy())
    
    # Calculate test metrics
    test_loss = test_loss / len(test_loader)
    test_acc = 100. * test_correct / test_total
    test_f1 = f1_score(test_true, test_preds, average='binary')
    
    print(f"📊 Test Set Results:")
    print(f"  Loss: {test_loss:.4f}")
    print(f"  Accuracy: {test_acc:.2f}%")
    print(f"  F1-Score: {test_f1:.4f}")
    
    # print(f"\n🔢 Confusion Matrix:")
    # cm = confusion_matrix(test_true, test_preds)
    # max_name_len = max(len(name) for name in class_names)
    # print(f"\n{' '*(max_name_len+2)}Predicted: {class_names[0]:<10s}  {class_names[1]:<10s}")
    # print(f"Actual {class_names[0]}:{' '*(max_name_len-len(class_names[0]))}{cm[0][0]:10d}  {cm[0][1]:10d}")
    # print(f"Actual {class_names[1]}:{' '*(max_name_len-len(class_names[1]))}{cm[1][0]:10d}  {cm[1][1]:10d}")
    
    # print(f"\n📋 Classification Report:")
    # print(classification_report(
    #     test_true, 
    #     test_preds,
    #     target_names=class_names,
    #     digits=4
    # ))
    
    # Save final model with metadata
    torch.save({
        'model_state_dict': model.state_dict(),
        'test_acc': test_acc,
        'test_f1': test_f1,
        'val_acc': best_val_acc,
        'val_f1': best_val_f1,
        'train_size': len(train_idx),
        'val_size': len(val_idx),
        'test_size': len(test_idx),
        'class_names': class_names
    }, f'{model_save_name}_final.pth')
    
    print(f"\n{'='*70}")
    print("COMPLETE!")
    print(f"{'='*70}")
    print(f"\n📁 Saved models:")
    print(f"  - best_{model_save_name}.pth (best validation)")
    print(f"  - {model_save_name}_final.pth (with test results)")
    print(f"\n📊 Final Performance:")
    print(f"  Validation: {best_val_acc:.2f}% (F1: {best_val_f1:.4f})")
    print(f"  Test:       {test_acc:.2f}% (F1: {test_f1:.4f})")
    print(f"{'='*70}\n")
    
    return model, test_idx

   








In [11]:
def train_stage1_als_detection():
    """Stage 1: Healthy vs ALS"""
    
    spectrogram_path = '/workspace/task1/mel_spectrograms'
    label_file = '/workspace/file_labels.csv'
    
    dataset = ALSDataset(spectrogram_path, label_file, use_phonation='all')
    
    return train_binary_classifier(
        dataset=dataset,
        stage_name="STAGE 1: ALS DETECTION",
        model_save_name="stage1_als_detector",
        class_names=['Healthy', 'ALS'],
        label_dim=2,
        batch_size=BATCH_SIZE,
        num_epochs=NUM_EPOCHS,
        learning_rate=LEARNING_RATE,
        device=DEVICE
    )




# ============================================================================
# USAGE
# ============================================================================

# Train Stage 1
print("\n" + "="*70)
print("STARTING STAGE 1 TRAINING")
print("="*70)
model_stage1, test_idx_stage1 = train_stage1_als_detection()


STARTING STAGE 1 TRAINING
Found 272 files in /workspace/task1/mel_spectrograms/phonationA
Found 272 files in /workspace/task1/mel_spectrograms/phonationE
Found 272 files in /workspace/task1/mel_spectrograms/phonationI
Found 272 files in /workspace/task1/mel_spectrograms/phonationO
Found 272 files in /workspace/task1/mel_spectrograms/phonationU
Found 272 files in /workspace/task1/mel_spectrograms/rhythmKA
Found 272 files in /workspace/task1/mel_spectrograms/rhythmPA
Found 272 files in /workspace/task1/mel_spectrograms/rhythmTA

=== Dataset Summary ===
Total samples: 2176
Class distribution:
  Class 1 (Severe dysarthria): 48 samples
  Class 2 (Moderate dysarthria): 208 samples
  Class 3 (Mild dysarthria): 456 samples
  Class 4 (ALS without dysarthria): 608 samples
  Class 5 (Healthy): 856 samples

STAGE 1: ALS DETECTION - DATASET SPLITTING
Total samples: 2176

📊 Split sizes:
  Train: 1523 samples (70.0%)
  Val:   326 samples (15.0%)
  Test:  327 samples (15.0%)

📊 Label distribution per

In [12]:
def train_stage2_dysarthria_detection():
    """Stage 2: ALS without dysarthria vs with dysarthria"""
    
    spectrogram_path = '/workspace/task1/mel_spectrograms'
    label_file = '/workspace/file_labels.csv'
    
    dataset = DysarthriaDetectionDataset(spectrogram_path, label_file, use_phonation='all')
    
    return train_binary_classifier(
        dataset=dataset,
        stage_name="STAGE 2: DYSARTHRIA DETECTION",
        model_save_name="stage2_dysarthria_detector",
        class_names=['No Dysarthria', 'Has Dysarthria'],
        label_dim=2,
        batch_size=BATCH_SIZE,
        num_epochs=NUM_EPOCHS,
        learning_rate=LEARNING_RATE,
        device=DEVICE
    )
# Train Stage 2
print("\n" + "="*70)
print("STARTING STAGE 2 TRAINING")
print("="*70)
model_stage2, test_idx_stage2 = train_stage2_dysarthria_detection()


STARTING STAGE 2 TRAINING
Found 272 files in /workspace/task1/mel_spectrograms/phonationA
Found 272 files in /workspace/task1/mel_spectrograms/phonationE
Found 272 files in /workspace/task1/mel_spectrograms/phonationI
Found 272 files in /workspace/task1/mel_spectrograms/phonationO
Found 272 files in /workspace/task1/mel_spectrograms/phonationU
Found 272 files in /workspace/task1/mel_spectrograms/rhythmKA
Found 272 files in /workspace/task1/mel_spectrograms/rhythmPA
Found 272 files in /workspace/task1/mel_spectrograms/rhythmTA

=== Dataset Summary ===
Total samples: 2176
Class distribution:
  Class 1 (Severe dysarthria): 48 samples
  Class 2 (Moderate dysarthria): 208 samples
  Class 3 (Mild dysarthria): 456 samples
  Class 4 (ALS without dysarthria): 608 samples
  Class 5 (Healthy): 856 samples

STAGE 2: DYSARTHRIA DETECTION - DATASET SPLITTING
Total samples: 1320

📊 Split sizes:
  Train: 924 samples (70.0%)
  Val:   198 samples (15.0%)
  Test:  198 samples (15.0%)

📊 Label distributi

In [ ]:
from sklearn.model_selection import train_test_split

def train_stage1_als_detection():
    """Stage 1: Healthy (856) vs ALS (1320)"""

    spectrogram_path = '/workspace/task1/mel_spectrograms'
    label_file = '/workspace/file_labels.csv'

    dataset = ALSDataset(spectrogram_path, label_file, use_phonation='all')

    print(f"\n{'='*70}")
    print("STAGE 1: ALS DETECTION - DATASET SPLITTING")
    print(f"{'='*70}")
    print(f"Total samples: {len(dataset)}")
    
    # ========================================================================
    # PROPER STRATIFIED TRAIN/VAL/TEST SPLIT
    # ========================================================================
    # Get all labels for stratification
    all_labels = [dataset[i][1] for i in range(len(dataset))]
    indices = list(range(len(dataset)))
    
    # First split: Train (70%) vs Temp (30%)
    train_idx, temp_idx = train_test_split(
        indices,
        test_size=0.3,
        stratify=all_labels,
        random_state=42
    )
    
    # Second split: Val (15%) vs Test (15%)
    temp_labels = [all_labels[i] for i in temp_idx]
    val_idx, test_idx = train_test_split(
        temp_idx,
        test_size=0.5,  # 0.5 of 30% = 15%
        stratify=temp_labels,
        random_state=42
    )
    
    # Show split statistics
    train_labels = [all_labels[i] for i in train_idx]
    val_labels = [all_labels[i] for i in val_idx]
    test_labels = [all_labels[i] for i in test_idx]
    
    print(f"\n📊 Split sizes:")
    print(f"  Train: {len(train_idx)} samples ({len(train_idx)/len(dataset)*100:.1f}%)")
    print(f"  Val:   {len(val_idx)} samples ({len(val_idx)/len(dataset)*100:.1f}%)")
    print(f"  Test:  {len(test_idx)} samples ({len(test_idx)/len(dataset)*100:.1f}%)")
    
    print(f"\n📊 Label distribution per split:")
    for label_val in sorted(set(all_labels)):
        train_count = train_labels.count(label_val)
        val_count = val_labels.count(label_val)
        test_count = test_labels.count(label_val)
        label_name = "Healthy" if label_val == 0 else "ALS"
        print(f"  {label_name} (Label {label_val}):")
        print(f"    Train: {train_count}, Val: {val_count}, Test: {test_count}")
    
    # Create dataset subsets
    train_dataset = Subset(dataset, train_idx)
    val_dataset = Subset(dataset, val_idx)
    test_dataset = Subset(dataset, test_idx)
    
    # Create data loaders
    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
    test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)
    
    print(f"\n✓ Data loaders created")
    print(f"{'='*70}\n")

    # Initialize model
    model = ASTModel(
        label_dim=2,
        fstride=10,
        tstride=10,
        input_fdim=128,
        input_tdim=1024,
        imagenet_pretrain=True,
        audioset_pretrain=False,
        model_size='base384'
    )
    model = model.to(DEVICE)

    # Loss and optimizer
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.SGD(
        model.parameters(),
        lr=LEARNING_RATE,
        momentum=0.9,  # Add momentum
        weight_decay=0.01  # Add regularization
    )

    scheduler = torch.optim.lr_scheduler.StepLR(
        optimizer,
        step_size=3,  # Reduce every 3 epochs
        gamma=0.5      # Multiply by 0.5
    )

    best_val_acc = 0.0

    # Training loop
    for epoch in range(NUM_EPOCHS):
        model.train()
        train_loss = 0
        train_correct = 0
        train_total = 0

        for batch_idx, (spectrograms, labels) in enumerate(train_loader):
            spectrograms = spectrograms.to(DEVICE)
            labels = labels.to(DEVICE)

            # Forward pass
            outputs = model(spectrograms)
            loss = criterion(outputs, labels)

            # Backward pass
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            # Stats
            train_loss += loss.item()
            _, predicted = outputs.max(1)
            train_total += labels.size(0)
            train_correct += predicted.eq(labels).sum().item()

            if batch_idx % 10 == 0:
                print(f'Epoch {epoch+1}, Batch {batch_idx}, Loss: {loss.item():.4f}')

        # calculate training metrics
        train_loss = train_loss / len(train_loader)
        train_acc = 100. * train_correct / train_total

        # Validation
        model.eval()
        val_loss = 0
        val_correct = 0
        val_total = 0

        with torch.no_grad():
            for spectrograms, labels in val_loader:
                spectrograms = spectrograms.to(DEVICE)
                labels = labels.to(DEVICE)

                outputs = model(spectrograms)
                loss = criterion(outputs, labels)

                val_loss += loss.item()
                _, predicted = outputs.max(1)
                val_total += labels.size(0)
                val_correct += predicted.eq(labels).sum().item()

        # Calculate validation metrics
        val_loss = val_loss / len(val_loader)
        val_acc = 100. * val_correct / val_total

        # Step the scheduler
        current_lr = optimizer.param_groups[0]['lr']
        scheduler.step()

        # Print epoch summary
        print(f'\n{"="*60}')
        print(f'Epoch {epoch+1}/{NUM_EPOCHS} Summary:')
        print(f'Learning Rate: {current_lr:.6f}')
        print(f'Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.2f}%')
        print(f'Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.2f}%')
        print(f'{"="*60}\n')

        # Save checkpoint
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            torch.save(model.state_dict(), 'best_model.pth')
            print(f'Saved new best model with validation accuracy: {val_acc:.2f}%\n')

    print("Training complete!")

    # Train...
    torch.save(model.state_dict(), 'stage1_als_detector.pth')

train_stage1_als_detection()

#Stage 2: Train Dysarthria Detection

In [ ]:
def train_stage2_dysarthria_detection():
    """Stage 2: ALS without dysarthria (608) vs with dysarthria (712)"""
    spectrogram_path = '/content/mel_spectrograms'
    label_file = '/content/file_labels.csv'

    dataset = DysarthriaDetectionDataset(spectrogram_path, label_file, use_phonation='all')

    train_size = int(0.8 * len(dataset))
    val_size = len(dataset) - train_size
    train_dataset, val_dataset = random_split(
        dataset,
        [train_size, val_size],
        generator=torch.Generator().manual_seed(42)
    )

    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE)

    # Initialize model
    model = ASTModel(
        label_dim=2,
        fstride=10,
        tstride=10,
        input_fdim=128,
        input_tdim=1024,
        imagenet_pretrain=True,
        audioset_pretrain=False,
        model_size='base384'
    )
    model = model.to(DEVICE)

    # Loss and optimizer
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.SGD(
        model.parameters(),
        lr=LEARNING_RATE,
        momentum=0.9,  # Add momentum
        weight_decay=0.01  # Add regularization
    )

    scheduler = torch.optim.lr_scheduler.StepLR(
        optimizer,
        step_size=3,  # Reduce every 3 epochs
        gamma=0.5      # Multiply by 0.5
    )

    best_val_acc = 0.0

    # Training loop
    for epoch in range(NUM_EPOCHS):
        model.train()
        train_loss = 0
        train_correct = 0
        train_total = 0

        for batch_idx, (spectrograms, labels) in enumerate(train_loader):
            spectrograms = spectrograms.to(DEVICE)
            labels = labels.to(DEVICE)

            # Forward pass
            outputs = model(spectrograms)
            loss = criterion(outputs, labels)

            # Backward pass
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            # Stats
            train_loss += loss.item()
            _, predicted = outputs.max(1)
            train_total += labels.size(0)
            train_correct += predicted.eq(labels).sum().item()

            if batch_idx % 10 == 0:
                print(f'Epoch {epoch+1}, Batch {batch_idx}, Loss: {loss.item():.4f}')

        # calculate training metrics
        train_loss = train_loss / len(train_loader)
        train_acc = 100. * train_correct / train_total

        # Validation
        model.eval()
        val_loss = 0
        val_correct = 0
        val_total = 0

        with torch.no_grad():
            for spectrograms, labels in val_loader:
                spectrograms = spectrograms.to(DEVICE)
                labels = labels.to(DEVICE)

                outputs = model(spectrograms)
                loss = criterion(outputs, labels)

                val_loss += loss.item()
                _, predicted = outputs.max(1)
                val_total += labels.size(0)
                val_correct += predicted.eq(labels).sum().item()

        # Calculate validation metrics
        val_loss = val_loss / len(val_loader)
        val_acc = 100. * val_correct / val_total

        # Step the scheduler
        current_lr = optimizer.param_groups[0]['lr']
        scheduler.step()

        # Print epoch summary
        print(f'\n{"="*60}')
        print(f'Epoch {epoch+1}/{NUM_EPOCHS} Summary:')
        print(f'Learning Rate: {current_lr:.6f}')
        print(f'Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.2f}%')
        print(f'Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.2f}%')
        print(f'{"="*60}\n')

        # Save checkpoint
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            torch.save(model.state_dict(), 'best_model.pth')
            print(f'Saved new best model with validation accuracy: {val_acc:.2f}%\n')

    print("Training complete!")

    torch.save(model.state_dict(), 'stage2_dysarthria_detector.pth')

train_stage2_dysarthria_detection()

In [ ]:
!cp /content/stage1_als_detector.pth /content/drive/MyDrive/
!cp /content/stage2_dysarthria_detector.pth /content/drive/MyDrive/

#Stage 3: Severity Classification

In [11]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Subset
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, f1_score
import numpy as np

NUM_EPOCHS = 15
BATCH_SIZE = 8
LEARNING_RATE = 1e-4

# ============================================================================
# GENERIC MULTI-CLASS TRAINING FUNCTION
# ============================================================================
def train_multiclass_classifier(
    dataset,
    stage_name,
    model_save_name,
    class_names,
    label_dim=3,
    batch_size=16,
    num_epochs=20,
    learning_rate=1e-4,
    device=None,
    use_class_weights=False
):
    """
    Generic training function for multi-class classification
    
    Parameters:
    -----------
    dataset : Dataset
        The dataset to train on
    stage_name : str
        Name of the stage (e.g., "STAGE 3: SEVERITY CLASSIFICATION")
    model_save_name : str
        Base name for saving models (e.g., "stage3_severity_classifier")
    class_names : dict
        Dictionary mapping label indices to names {0: 'Class0', 1: 'Class1', ...}
    label_dim : int
        Number of output classes
    batch_size : int
        Batch size for training
    num_epochs : int
        Number of training epochs
    learning_rate : float
        Learning rate
    device : torch.device
        Device to train on (default: auto-detect)
    use_class_weights : bool
        Whether to use class weights in loss function
    
    Returns:
    --------
    model : ASTModel
        Trained model
    test_idx : list
        Test set indices
    """
    
    if device is None:
        device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    
    print(f"\n{'='*70}")
    print(f"{stage_name} - LABEL VALIDATION & DATASET SPLITTING")
    print(f"{'='*70}")
    
    # ========================================================================
    # LABEL VALIDATION
    # ========================================================================
    print("\n📋 Validating all labels...")
    all_labels = [dataset[i][1] for i in range(len(dataset))]
    
    min_label = min(all_labels)
    max_label = max(all_labels)
    unique_labels = sorted(set(all_labels))
    
    print(f"  Min label: {min_label}")
    print(f"  Max label: {max_label}")
    print(f"  Unique labels: {unique_labels}")
    print(f"\n  Label distribution:")
    for label_val in unique_labels:
        count = all_labels.count(label_val)
        label_name = class_names.get(label_val, f"Unknown-{label_val}")
        print(f"    Label {label_val} ({label_name}): {count} samples")
    
    # Validation check
    expected_labels = set(range(label_dim))
    if set(unique_labels) != expected_labels:
        raise ValueError(
            f"❌ INVALID LABELS! Expected {expected_labels}, got {set(unique_labels)}"
        )
    
    print("\n✓ All labels valid!")
    print(f"Total samples: {len(dataset)}")
    
    # ========================================================================
    # PROPER STRATIFIED TRAIN/VAL/TEST SPLIT
    # ========================================================================
    indices = list(range(len(dataset)))
    
    # First split: Train (70%) vs Temp (30%)
    train_idx, temp_idx = train_test_split(
        indices,
        test_size=0.3,
        stratify=all_labels,
        random_state=42
    )
    
    # Second split: Val (15%) vs Test (15%)
    temp_labels = [all_labels[i] for i in temp_idx]
    val_idx, test_idx = train_test_split(
        temp_idx,
        test_size=0.5,
        stratify=temp_labels,
        random_state=42
    )
    
    # Show split statistics
    train_labels = [all_labels[i] for i in train_idx]
    val_labels = [all_labels[i] for i in val_idx]
    test_labels = [all_labels[i] for i in test_idx]
    
    print(f"\n📊 Split sizes:")
    print(f"  Train: {len(train_idx)} samples ({len(train_idx)/len(dataset)*100:.1f}%)")
    print(f"  Val:   {len(val_idx)} samples ({len(val_idx)/len(dataset)*100:.1f}%)")
    print(f"  Test:  {len(test_idx)} samples ({len(test_idx)/len(dataset)*100:.1f}%)")
    
    print(f"\n📊 Label distribution per split:")
    for label_val in sorted(unique_labels):
        train_count = train_labels.count(label_val)
        val_count = val_labels.count(label_val)
        test_count = test_labels.count(label_val)
        label_name = class_names.get(label_val, f"Class {label_val}")
        print(f"  {label_name}:")
        print(f"    Train: {train_count}, Val: {val_count}, Test: {test_count}")
    
    # Create dataset subsets
    train_dataset = Subset(dataset, train_idx)
    val_dataset = Subset(dataset, val_idx)
    test_dataset = Subset(dataset, test_idx)
    
    # Create data loaders
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)  # ✅ Fixed: shuffle=True (not 'True')
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)
    
    print(f"\n✓ Data loaders created")
    print(f"{'='*70}\n")
    
    # ========================================================================
    # MODEL INITIALIZATION
    # ========================================================================
    model = ASTModel(
        label_dim=label_dim,
        fstride=10,
        tstride=10,
        input_fdim=128,
        input_tdim=1024,
        imagenet_pretrain=True,
        audioset_pretrain=False,
        model_size='base384'
    )
    model = model.to(device)
    
    # Loss function with optional class weights
    if use_class_weights:
        # Calculate inverse frequency weights
        label_counts = torch.tensor([train_labels.count(i) for i in range(label_dim)], dtype=torch.float32)
        class_weights = 1.0 / label_counts
        class_weights = class_weights / class_weights.sum() * label_dim
        class_weights = class_weights.to(device)
        
        print("Using class weights:")
        for i, weight in enumerate(class_weights):
            print(f"  {class_names[i]}: {weight:.4f}")
        print()
        
        criterion = nn.CrossEntropyLoss(weight=class_weights)
    else:
        criterion = nn.CrossEntropyLoss()
    
    # Optimizer and scheduler
    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=learning_rate,
        weight_decay=0.01
    )
    
    scheduler = torch.optim.lr_scheduler.StepLR(
        optimizer,
        step_size=3,
        gamma=0.5
    )
    
    best_val_acc = 0.0
    best_val_f1 = 0.0
    
    # ========================================================================
    # TRAINING LOOP
    # ========================================================================
    print(f"{'='*70}")
    print("TRAINING")
    print(f"{'='*70}\n")
    
    for epoch in range(num_epochs):
        # Training phase
        model.train()
        train_loss = 0
        train_correct = 0
        train_total = 0
        
        for batch_idx, (spectrograms, labels) in enumerate(train_loader):
            spectrograms = spectrograms.to(device)
            labels = labels.to(device)
            
            # Forward pass
            outputs = model(spectrograms)
            loss = criterion(outputs, labels)
            
            # Backward pass
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            
            # Stats
            train_loss += loss.item()
            _, predicted = outputs.max(1)
            train_total += labels.size(0)
            train_correct += predicted.eq(labels).sum().item()
            
            if batch_idx % 10 == 0:
                print(f'Epoch {epoch+1}/{num_epochs}, Batch {batch_idx}, Loss: {loss.item():.4f}')
        
        # Calculate training metrics
        train_loss = train_loss / len(train_loader)
        train_acc = 100. * train_correct / train_total
        
        # Validation phase
        model.eval()
        val_loss = 0
        val_correct = 0
        val_total = 0
        val_preds = []
        val_true = []
        
        with torch.no_grad():
            for spectrograms, labels in val_loader:
                spectrograms = spectrograms.to(device)
                labels = labels.to(device)
                
                outputs = model(spectrograms)
                loss = criterion(outputs, labels)
                
                val_loss += loss.item()
                _, predicted = outputs.max(1)
                val_total += labels.size(0)
                val_correct += predicted.eq(labels).sum().item()
                
                val_preds.extend(predicted.cpu().numpy())
                val_true.extend(labels.cpu().numpy())
        
        # Calculate validation metrics
        val_loss = val_loss / len(val_loader)
        val_acc = 100. * val_correct / val_total
        val_f1 = f1_score(val_true, val_preds, average='weighted')
        
        # Step the scheduler
        current_lr = optimizer.param_groups[0]['lr']
        scheduler.step()
        
        # Print epoch summary
        print(f'\n{"="*60}')
        print(f'Epoch {epoch+1}/{num_epochs} Summary:')
        print(f'Learning Rate: {current_lr:.6f}')
        print(f'Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.2f}%')
        print(f'Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.2f}% | Val F1: {val_f1:.4f}')
        print(f'{"="*60}\n')
        
        # Save best model based on F1 score
        if val_f1 > best_val_f1:
            best_val_f1 = val_f1
            best_val_acc = val_acc
            torch.save({
                'epoch': epoch,
                'model_state_dict': model.state_dict(),  # ✅ Fixed: save as dict
                'optimizer_state_dict': optimizer.state_dict(),
                'val_acc': val_acc,
                'val_f1': val_f1
            }, f'best_{model_save_name}.pth')
            print(f'💾 Saved new best model (Val F1: {val_f1:.4f})\n')
    
    print("Training complete!")
    
    # ========================================================================
    # FINAL TEST EVALUATION
    # ========================================================================
    print(f"\n{'='*70}")
    print("FINAL TEST SET EVALUATION")
    print(f"{'='*70}\n")
    
    # Load best model
    checkpoint = torch.load(f'best_{model_save_name}.pth')
    model.load_state_dict(checkpoint['model_state_dict'])  # ✅ Fixed: correct key
    print(f"✓ Loaded best model from epoch {checkpoint['epoch']+1}")
    print(f"  Val Accuracy: {checkpoint['val_acc']:.2f}%")
    print(f"  Val F1: {checkpoint['val_f1']:.4f}\n")
    
    # Evaluate on test set
    model.eval()
    test_loss = 0
    test_correct = 0
    test_total = 0
    test_preds = []
    test_true = []
    
    with torch.no_grad():
        for spectrograms, labels in test_loader:
            spectrograms = spectrograms.to(device)
            labels = labels.to(device)
            
            outputs = model(spectrograms)
            loss = criterion(outputs, labels)
            
            test_loss += loss.item()
            _, predicted = outputs.max(1)
            test_total += labels.size(0)
            test_correct += predicted.eq(labels).sum().item()
            
            test_preds.extend(predicted.cpu().numpy())
            test_true.extend(labels.cpu().numpy())
    
    # Calculate test metrics
    test_loss = test_loss / len(test_loader)
    test_acc = 100. * test_correct / test_total
    test_f1 = f1_score(test_true, test_preds, average='weighted')
    
    print(f"📊 Test Set Results:")
    print(f"  Loss: {test_loss:.4f}")
    print(f"  Accuracy: {test_acc:.2f}%")
    print(f"  F1-Score: {test_f1:.4f}")
    
    print(f"\n🔢 Confusion Matrix:")
    cm = confusion_matrix(test_true, test_preds, labels=list(range(label_dim)))
    print("\n          Predicted →")
    print("Actual ↓  ", "  ".join([f"{class_names[i]:9s}" for i in range(label_dim)]))
    for i in range(label_dim):
        print(f"{class_names[i]:9s}", "  ".join([f"{cm[i][j]:9d}" for j in range(label_dim)]))
    
    print(f"\n📋 Classification Report:")
    print(classification_report(
        test_true,
        test_preds,
        labels=list(range(label_dim)),
        target_names=[class_names[i] for i in range(label_dim)],
        digits=4
    ))
    
    # Save final model with metadata
    torch.save({
        'model_state_dict': model.state_dict(),
        'test_acc': test_acc,
        'test_f1': test_f1,
        'val_acc': best_val_acc,
        'val_f1': best_val_f1,
        'train_size': len(train_idx),
        'val_size': len(val_idx),
        'test_size': len(test_idx),
        'class_names': class_names
    }, f'{model_save_name}_final.pth')
    
    print(f"\n{'='*70}")
    print("COMPLETE!")
    print(f"{'='*70}")
    print(f"\n📁 Saved models:")
    print(f"  - best_{model_save_name}.pth (best validation)")
    print(f"  - {model_save_name}_final.pth (with test results)")
    print(f"\n📊 Final Performance:")
    print(f"  Validation: {best_val_acc:.2f}% (F1: {best_val_f1:.4f})")
    print(f"  Test:       {test_acc:.2f}% (F1: {test_f1:.4f})")
    print(f"{'='*70}\n")
    
    return model, test_idx


# ============================================================================
# STAGE 3 WRAPPER FUNCTION
# ============================================================================

# Define label names (as per your earlier specification)
LABEL_NAMES = {
    0: 'Severe',    # Was class 1 in CSV
    1: 'Moderate',  # Was class 2 in CSV
    2: 'Mild'       # Was class 3 in CSV
}

def train_stage3_severity_classification():
    """Stage 3: Severity Classification (Severe/Moderate/Mild)"""
    
    spectrogram_path = '/workspace/task1/mel_spectrograms'
    label_file = '/workspace/file_labels.csv'
    
    dataset = SeverityDataset(spectrogram_path, label_file, use_phonation='all')
    
    return train_multiclass_classifier(
        dataset=dataset,
        stage_name="STAGE 3: SEVERITY CLASSIFICATION",
        model_save_name="stage3_severity_classifier",
        class_names=LABEL_NAMES,
        label_dim=3,
        batch_size=BATCH_SIZE,
        num_epochs=NUM_EPOCHS,
        learning_rate=LEARNING_RATE,
        device=DEVICE,
        use_class_weights=False  # Set to True if you want to use class weights
    )


# ============================================================================
# USAGE
# ============================================================================

model, test_indices = train_stage3_severity_classification()

Found 272 files in /workspace/task1/mel_spectrograms/phonationA
Found 272 files in /workspace/task1/mel_spectrograms/phonationE
Found 272 files in /workspace/task1/mel_spectrograms/phonationI
Found 272 files in /workspace/task1/mel_spectrograms/phonationO
Found 272 files in /workspace/task1/mel_spectrograms/phonationU
Found 272 files in /workspace/task1/mel_spectrograms/rhythmKA
Found 272 files in /workspace/task1/mel_spectrograms/rhythmPA
Found 272 files in /workspace/task1/mel_spectrograms/rhythmTA

=== Dataset Summary ===
Total samples: 2176
Class distribution:
  Class 1 (Severe dysarthria): 48 samples
  Class 2 (Moderate dysarthria): 208 samples
  Class 3 (Mild dysarthria): 456 samples
  Class 4 (ALS without dysarthria): 608 samples
  Class 5 (Healthy): 856 samples

STAGE 3: SEVERITY CLASSIFICATION - LABEL VALIDATION & DATASET SPLITTING

📋 Validating all labels...
  Min label: 0
  Max label: 2
  Unique labels: [0, 1, 2]

  Label distribution:
    Label 0 (Severe): 48 samples
    La

In [13]:
import pandas as pd

from dataset import DysarthriaDataset

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Subset
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, f1_score
import numpy as np

NUM_EPOCHS = 15
LEARNING_RATE = 1e-4
BATCH_SIZE = 8

from sklearn.metrics import (
    classification_report, 
    confusion_matrix, 
    accuracy_score,
    f1_score,
    precision_recall_fscore_support
)

LABEL_NAMES = {
    0: 'Severe',    # Was class 1 in CSV
    1: 'Moderate',  # Was class 2 in CSV
    2: 'Mild'       # Was class 3 in CSV
}

# For hierarchical model final output
HIERARCHICAL_LABELS = {
    0: 'ALS-Severe-Dysarthria',
    1: 'ALS-Moderate-Dysarthria',
    2: 'ALS-Mild-Dysarthria',
    3: 'ALS-No-Dysarthria',
    4: 'Healthy',
}

CSV_TO_HIERARCHICAL = {
    1: 0,  # Severe dysarthria → 0
    2: 1,  # Moderate dysarthria → 1
    3: 2,  # Mild dysarthria → 2
    4: 3,  # ALS-No-Dysarthria → 3
    5: 4   # Healthy → 4
}

def load_model(checkpoint_path, label_dim, device=DEVICE):
    """Load a trained model from checkpoint"""
    print(f"Loading model from: {checkpoint_path}")
    
    model = ASTModel(
        label_dim=label_dim,
        fstride=10,
        tstride=10,
        input_fdim=128,
        input_tdim=1024,
        imagenet_pretrain=False,
        audioset_pretrain=False,
        model_size='base384'
    )
    
    checkpoint = torch.load(checkpoint_path, map_location=device)
    
    if isinstance(checkpoint, dict) and 'model_state_dict' in checkpoint:
        model.load_state_dict(checkpoint['model_state_dict'])
        if 'val_f1' in checkpoint:
            print(f"  ✓ Val F1: {checkpoint['val_f1']:.4f}")
    else:
        model.load_state_dict(checkpoint)
    
    model = model.to(device)
    model.eval()
    
    return model

def create_unified_test_set(spectrogram_path, label_file):
    """
    Create a unified test set that works for all 3 stages
    by splitting the FULL dataset once and filtering per stage
    """
    print(f"\n{'='*70}")
    print("CREATING UNIFIED TEST SET")
    print(f"{'='*70}\n")
    
    # Load full CSV
    df = pd.read_csv(label_file)
    
    # Get all sample indices
    all_indices = list(range(len(df)))
    all_labels = df['Class'].tolist()
    
    # Split: 70% train, 15% val, 15% test
    train_val_idx, test_idx = train_test_split(
        all_indices,
        test_size=0.15,
        stratify=all_labels,
        random_state=42
    )
    
    # Get test samples
    test_df = df.iloc[test_idx].copy()
    test_df['hierarchical_label'] = test_df['Class'].map(CSV_TO_HIERARCHICAL)
    
    print(f"✓ Created unified test set: {len(test_df)} samples")
    print(f"\nTest set distribution:")
    for csv_class in sorted(test_df['Class'].unique()):
        count = (test_df['Class'] == csv_class).sum()
        hier_label = CSV_TO_HIERARCHICAL[csv_class]
        print(f"  CSV Class {csv_class} → {HIERARCHICAL_LABELS[hier_label]}: {count} samples")
    
    return test_df, test_idx

# ============================================================================
# HIERARCHICAL EVALUATION FUNCTION
# ============================================================================
def evaluate_hierarchical_model(
    test_df,
    spectrogram_path,
    als_model_path,
    dysarthria_model_path,
    severity_model_path,
    device=DEVICE
):
    """
    Evaluate the complete hierarchical model on test set
    """
    print(f"\n{'='*70}")
    print("HIERARCHICAL MODEL EVALUATION")
    print(f"{'='*70}\n")
    
    # Load all three models
    print("Loading models...")
    als_model = load_model(als_model_path, label_dim=2, device=device)
    dysarthria_model = load_model(dysarthria_model_path, label_dim=2, device=device)
    severity_model = load_model(severity_model_path, label_dim=3, device=device)
    print("✓ All models loaded\n")
    
    # Prepare test data
    # For simplicity, we'll create a simple dataset that loads spectrograms by ID
    # You'll need to adapt this to your actual dataset structure
    
    test_ids = test_df['ID'].tolist()
    ground_truth = test_df['hierarchical_label'].tolist()
    
    all_predictions = []
    all_confidences = []
    
    # Per-stage tracking
    stage1_preds = []
    stage1_truth = []
    stage2_preds = []
    stage2_truth = []
    stage3_preds = []
    stage3_truth = []
    
    print(f"Evaluating {len(test_ids)} samples...")
    
    # Load your base dataset to access spectrograms
    # This is a simplified example - adapt to your dataset structure
    full_dataset = DysarthriaDataset(spectrogram_path, label_file='/workspace/file_labels.csv', use_phonation='all')
    
    with torch.no_grad():
        for idx, (patient_id, true_hier_label) in enumerate(zip(test_ids, ground_truth)):
            
            # Find the spectrogram for this patient
            # This is dataset-specific - you may need to adjust
            # For now, assuming we can access by index
            spec_idx = None
            for i in range(len(full_dataset)):
                if full_dataset.get_patient_id(i) == patient_id:
                    spec_idx = i
                    break
            spectrogram, _ = full_dataset[spec_idx]
            spectrogram = spectrogram.unsqueeze(0).to(device)
            
            # ================================================================
            # STAGE 1: Healthy vs ALS
            # ================================================================
            als_output = als_model(spectrogram)
            als_prob = torch.softmax(als_output, dim=1)
            als_pred = torch.argmax(als_prob, dim=1).item()
            
            # Track Stage 1
            stage1_truth.append(0 if true_hier_label == 4 else 1)
            stage1_preds.append(als_pred)
            
            if als_pred == 0:  # Predicted Healthy
                final_pred = 4
                confidence = als_prob[0][0].item()
            
            else:  # Predicted ALS
                # ============================================================
                # STAGE 2: No Dysarthria vs Has Dysarthria
                # ============================================================
                dysarthria_output = dysarthria_model(spectrogram)
                dysarthria_prob = torch.softmax(dysarthria_output, dim=1)
                dysarthria_pred = torch.argmax(dysarthria_prob, dim=1).item()
                
                # Track Stage 2 (only for true ALS patients)
                if true_hier_label <= 3:
                    stage2_truth.append(0 if true_hier_label == 3 else 1)
                    stage2_preds.append(dysarthria_pred)
                
                if dysarthria_pred == 0:  # No Dysarthria
                    final_pred = 3
                    confidence = dysarthria_prob[0][0].item()
                
                else:  # Has Dysarthria
                    # ========================================================
                    # STAGE 3: Severe/Moderate/Mild
                    # ========================================================
                    severity_output = severity_model(spectrogram)
                    severity_prob = torch.softmax(severity_output, dim=1)
                    severity_pred = torch.argmax(severity_prob, dim=1).item()
                    
                    # Track Stage 3 (only for true dysarthric patients)
                    if true_hier_label <= 2:
                        stage3_truth.append(true_hier_label)
                        stage3_preds.append(severity_pred)
                    
                    # Map to hierarchical label (0=Severe, 1=Moderate, 2=Mild)
                    final_pred = severity_pred
                    confidence = severity_prob[0][severity_pred].item()
            
            all_predictions.append(final_pred)
            all_confidences.append(confidence)
            
            if (idx + 1) % 20 == 0:
                print(f"  Processed {idx + 1}/{len(test_ids)} samples...")
    
    print(f"✓ Evaluation complete!\n")
    
    # ========================================================================
    # OVERALL PERFORMANCE
    # ========================================================================
    overall_acc = accuracy_score(ground_truth, all_predictions)
    overall_f1 = f1_score(ground_truth, all_predictions, average='weighted', zero_division=0)
    avg_conf = np.mean(all_confidences)
    
    print("="*70)
    print("OVERALL HIERARCHICAL MODEL PERFORMANCE")
    print("="*70)
    print(f"\n📊 Summary:")
    print(f"  Accuracy: {overall_acc:.4f} ({overall_acc*100:.2f}%)")
    print(f"  Weighted F1: {overall_f1:.4f}")
    print(f"  Avg Confidence: {avg_conf:.4f}")
    
    # Confusion Matrix
    print(f"\n🔢 Confusion Matrix:")
    cm = confusion_matrix(ground_truth, all_predictions, labels=[0,1,2,3,4])
    
    print("\n                Predicted →")
    header = "Actual ↓        "
    for i in range(5):
        header += f"{HIERARCHICAL_LABELS[i][:15]:15s}  "
    print(header)
    
    for i in range(5):
        row = f"{HIERARCHICAL_LABELS[i][:15]:15s}"
        for j in range(5):
            row += f"{cm[i][j]:15d}  "
        print(row)
    
    # Classification Report
    print(f"\n📋 Detailed Per-Class Report:")
    print(classification_report(
        ground_truth,
        all_predictions,
        labels=[0,1,2,3,4],
        target_names=[HIERARCHICAL_LABELS[i] for i in range(5)],
        digits=4,
        zero_division=0
    ))
    
    # ========================================================================
    # PER-STAGE BREAKDOWN
    # ========================================================================
    print("\n" + "="*70)
    print("PER-STAGE PERFORMANCE BREAKDOWN")
    print("="*70)
    
    # Stage 1
    if len(stage1_truth) > 0:
        stage1_acc = accuracy_score(stage1_truth, stage1_preds)
        stage1_f1 = f1_score(stage1_truth, stage1_preds, average='binary', zero_division=0)
        
        print(f"\n🎯 STAGE 1 - ALS Detection:")
        print(f"  Accuracy: {stage1_acc:.4f} ({stage1_acc*100:.2f}%)")
        print(f"  F1-Score: {stage1_f1:.4f}")
        print(f"  Samples: {len(stage1_truth)}")
        
        if len(set(stage1_truth)) > 1:
            s1_cm = confusion_matrix(stage1_truth, stage1_preds, labels=[0,1])
            print(f"\n  Confusion Matrix:")
            print(f"                  Predicted: Healthy   ALS")
            print(f"  Actual Healthy:          {s1_cm[0][0]:7d}  {s1_cm[0][1]:7d}")
            if len(s1_cm) > 1:
                print(f"  Actual ALS:              {s1_cm[1][0]:7d}  {s1_cm[1][1]:7d}")
    
    # Stage 2
    if len(stage2_truth) > 0:
        stage2_acc = accuracy_score(stage2_truth, stage2_preds)
        stage2_f1 = f1_score(stage2_truth, stage2_preds, average='binary', zero_division=0)
        
        print(f"\n🎯 STAGE 2 - Dysarthria Detection:")
        print(f"  Accuracy: {stage2_acc:.4f} ({stage2_acc*100:.2f}%)")
        print(f"  F1-Score: {stage2_f1:.4f}")
        print(f"  Samples: {len(stage2_truth)}")
        
        if len(set(stage2_truth)) > 1:
            s2_cm = confusion_matrix(stage2_truth, stage2_preds, labels=[0,1])
            print(f"\n  Confusion Matrix:")
            print(f"                  Predicted: No-Dys  Has-Dys")
            print(f"  Actual No-Dys:           {s2_cm[0][0]:7d}  {s2_cm[0][1]:7d}")
            if len(s2_cm) > 1:
                print(f"  Actual Has-Dys:          {s2_cm[1][0]:7d}  {s2_cm[1][1]:7d}")
    
    # Stage 3
    if len(stage3_truth) > 0:
        stage3_acc = accuracy_score(stage3_truth, stage3_preds)
        stage3_f1 = f1_score(stage3_truth, stage3_preds, average='weighted', zero_division=0)
        
        print(f"\n🎯 STAGE 3 - Severity Classification:")
        print(f"  Accuracy: {stage3_acc:.4f} ({stage3_acc*100:.2f}%)")
        print(f"  Weighted F1: {stage3_f1:.4f}")
        print(f"  Samples: {len(stage3_truth)}")
        
        print(f"\n  Per-Severity Performance:")
        print(classification_report(
            stage3_truth,
            stage3_preds,
            labels=[0,1,2],
            target_names=['Severe', 'Moderate', 'Mild'],
            digits=4,
            zero_division=0
        ))
    
    print("="*70)
    
    # ========================================================================
    # SAVE RESULTS
    # ========================================================================
    results_df = pd.DataFrame({
        'ID': test_ids,
        'ground_truth': ground_truth,
        'prediction': all_predictions,
        'confidence': all_confidences,
        'ground_truth_label': [HIERARCHICAL_LABELS[i] for i in ground_truth],
        'prediction_label': [HIERARCHICAL_LABELS[i] for i in all_predictions],
        'correct': [g == p for g, p in zip(ground_truth, all_predictions)]
    })
    
    results_df.to_csv('hierarchical_evaluation_results.csv', index=False)
    print(f"\n💾 Saved detailed results to: hierarchical_evaluation_results.csv")
    
    return {
        'overall_accuracy': overall_acc,
        'overall_f1': overall_f1,
        'stage1_accuracy': stage1_acc if len(stage1_truth) > 0 else None,
        'stage2_accuracy': stage2_acc if len(stage2_truth) > 0 else None,
        'stage3_accuracy': stage3_acc if len(stage3_truth) > 0 else None,
        'confusion_matrix': cm
    }

def main():
    """
    Complete pipeline: Train all 3 stages and evaluate hierarchically
    """
    
    print("\n" + "="*70)
    print("HIERARCHICAL ALS DYSARTHRIA SEVERITY CLASSIFICATION")
    print("="*70)
    print(f"Device: {DEVICE}")
    print(f"Batch Size: {BATCH_SIZE}")
    print(f"Epochs: {NUM_EPOCHS}")
    print(f"Learning Rate: {LEARNING_RATE}")
    print("="*70)
    
    spectrogram_path = '/workspace/task1/mel_spectrograms'
    label_file = '/workspace/file_labels.csv'

    print("\n\n" + "🎯 "*35)
    print("STEP 4: CREATING UNIFIED TEST SET")
    print("🎯 "*35 + "\n")
    
    test_df, test_indices = create_unified_test_set(spectrogram_path, label_file)
    
    # ========================================================================
    # STEP 5: HIERARCHICAL EVALUATION
    # ========================================================================
    print("\n\n" + "📊 "*35)
    print("STEP 5: HIERARCHICAL MODEL EVALUATION")
    print("📊 "*35 + "\n")
    
    results = evaluate_hierarchical_model(
        test_df=test_df,
        spectrogram_path=spectrogram_path,
        als_model_path='best_stage1_als_detector.pth',
        dysarthria_model_path='best_stage2_dysarthria_detector.pth',
        severity_model_path='best_stage3_severity_classifier.pth',
        device=DEVICE
    )
    
    # ========================================================================
    # FINAL SUMMARY
    # ========================================================================
    print("\n\n" + "="*70)
    print("🎉 TRAINING AND EVALUATION COMPLETE! 🎉")
    print("="*70)
    print(f"\n📊 FINAL RESULTS:")
    print(f"  Overall Accuracy: {results['overall_accuracy']:.4f} ({results['overall_accuracy']*100:.2f}%)")
    print(f"  Overall F1-Score: {results['overall_f1']:.4f}")
    print(f"\n  Stage 1 Accuracy: {results['stage1_accuracy']:.4f}")
    print(f"  Stage 2 Accuracy: {results['stage2_accuracy']:.4f}")
    print(f"  Stage 3 Accuracy: {results['stage3_accuracy']:.4f}")
    print(f"\n📁 Saved Models:")
    print(f"  - best_stage1_als_detector.pth")
    print(f"  - best_stage2_dysarthria_detector.pth")
    print(f"  - best_stage3_severity_classifier.pth")
    print(f"\n📄 Results:")
    print(f"  - hierarchical_evaluation_results.csv")
    print("="*70 + "\n")

main()


HIERARCHICAL ALS DYSARTHRIA SEVERITY CLASSIFICATION
Device: cuda
Batch Size: 8
Epochs: 15
Learning Rate: 0.0001


🎯 🎯 🎯 🎯 🎯 🎯 🎯 🎯 🎯 🎯 🎯 🎯 🎯 🎯 🎯 🎯 🎯 🎯 🎯 🎯 🎯 🎯 🎯 🎯 🎯 🎯 🎯 🎯 🎯 🎯 🎯 🎯 🎯 🎯 🎯 
STEP 4: CREATING UNIFIED TEST SET
🎯 🎯 🎯 🎯 🎯 🎯 🎯 🎯 🎯 🎯 🎯 🎯 🎯 🎯 🎯 🎯 🎯 🎯 🎯 🎯 🎯 🎯 🎯 🎯 🎯 🎯 🎯 🎯 🎯 🎯 🎯 🎯 🎯 🎯 🎯 


CREATING UNIFIED TEST SET

✓ Created unified test set: 41 samples

Test set distribution:
  CSV Class 1 → ALS-Severe-Dysarthria: 1 samples
  CSV Class 2 → ALS-Moderate-Dysarthria: 4 samples
  CSV Class 3 → ALS-Mild-Dysarthria: 9 samples
  CSV Class 4 → ALS-No-Dysarthria: 11 samples
  CSV Class 5 → Healthy: 16 samples


📊 📊 📊 📊 📊 📊 📊 📊 📊 📊 📊 📊 📊 📊 📊 📊 📊 📊 📊 📊 📊 📊 📊 📊 📊 📊 📊 📊 📊 📊 📊 📊 📊 📊 📊 
STEP 5: HIERARCHICAL MODEL EVALUATION
📊 📊 📊 📊 📊 📊 📊 📊 📊 📊 📊 📊 📊 📊 📊 📊 📊 📊 📊 📊 📊 📊 📊 📊 📊 📊 📊 📊 📊 📊 📊 📊 📊 📊 📊 


HIERARCHICAL MODEL EVALUATION

Loading models...
Loading model from: best_stage1_als_detector.pth
---------------AST Model Summary---------------
ImageNet pretraining: False, AudioSet pretraining: False
f

In [9]:
from sklearn.metrics import classification_report, confusion_matrix, f1_score
from torch.utils.data import WeightedRandomSampler

def create_balanced_sampler(dataset):
    # Count samples per class
    labels = [dataset[i][1] for i in range(len(dataset))]
    class_counts = torch.bincount(torch.tensor(labels))
    
    # Weight inversely proportional to class frequency
    class_weights = 1.0 / class_counts.float()
    sample_weights = [class_weights[label] for label in labels]
    
    sampler = WeightedRandomSampler(
        weights=sample_weights,
        num_samples=len(dataset),
        replacement=True
    )
    return sampler

    
def train_stage3_severity_classification():
    """Stage 3: Mild (456) vs Moderate (208) vs Severe (48)"""

    spectrogram_path = '/workspace/task1/mel_spectrograms'
    label_file = '/workspace/file_labels.csv'

    dataset = SeverityDataset(spectrogram_path, label_file, use_phonation='all')

    # ✅ AGGRESSIVE VALIDATION: Check ALL labels before training
    print("\n=== VALIDATING ALL LABELS ===")
    all_labels = []
    for i in range(len(dataset)):
        _, label = dataset[i]
        all_labels.append(label)

    min_label = min(all_labels)
    max_label = max(all_labels)
    unique_labels = sorted(set(all_labels))

    print(f"Min label: {min_label}")
    print(f"Max label: {max_label}")
    print(f"Unique labels: {unique_labels}")
    print(f"Label distribution:")
    for label_val in unique_labels:
        count = all_labels.count(label_val)
        print(f"  Label {label_val}: {count} samples")

    # ❌ CRITICAL: Stop if invalid labels exist
    if min_label < 0 or max_label >= 3:
        raise ValueError(f"INVALID LABELS FOUND! Expected [0,1,2], got [{min_label},{max_label}]")

    print("✓ All labels valid!\n")

    train_size = int(0.8 * len(dataset))
    val_size = len(dataset) - train_size
    train_dataset, val_dataset = random_split(
        dataset,
        [train_size, val_size],
        generator=torch.Generator().manual_seed(42)
    )

    
    #train_sampler = create_balanced_sampler(train_dataset)


    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle = 'True')
    val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE)

    # Initialize model
    model = ASTModel(
        label_dim=3,
        fstride=10,
        tstride=10,
        input_fdim=128,
        input_tdim=1024,
        imagenet_pretrain=True,
        audioset_pretrain=False,
        model_size='base384'
    )
    model = model.to(DEVICE)


    # class_weights = torch.tensor([
    #     48/456,    # Weight for Severe (underrepresented)
    #     208/456,   # Weight for Moderate
    #     456/456    # Weight for Mild (most common)
    # ], dtype=torch.float32).to(DEVICE)

    # class_weights = class_weights.to(next(model.parameters()).dtype)

    # criterion = nn.CrossEntropyLoss(weight=class_weights)

    # Loss and optimizer
    criterion = nn.CrossEntropyLoss()  # No weights

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=LEARNING_RATE,
        weight_decay=0.01  # Add regularization
    )

    scheduler = torch.optim.lr_scheduler.StepLR(
        optimizer,
        step_size=3,  # Reduce every 3 epochs
        gamma=0.5      # Multiply by 0.5
    )

    best_val_acc = 0.0

    # Training loop
    for epoch in range(NUM_EPOCHS):
        model.train()
        train_loss = 0
        train_correct = 0
        train_total = 0

        for batch_idx, (spectrograms, labels) in enumerate(train_loader):
            spectrograms = spectrograms.to(DEVICE)
            labels = labels.to(DEVICE)

            # Forward pass
            outputs = model(spectrograms)
            loss = criterion(outputs, labels)

            # Backward pass
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            # Stats
            train_loss += loss.item()
            _, predicted = outputs.max(1)
            train_total += labels.size(0)
            train_correct += predicted.eq(labels).sum().item()

            if batch_idx % 10 == 0:
                print(f'Epoch {epoch+1}, Batch {batch_idx}, Loss: {loss.item():.4f}')

        # calculate training metrics
        train_loss = train_loss / len(train_loader)
        train_acc = 100. * train_correct / train_total

        # Validation
        model.eval()
        val_loss = 0
        val_correct = 0
        val_total = 0

        with torch.no_grad():
            for spectrograms, labels in val_loader:
                spectrograms = spectrograms.to(DEVICE)
                labels = labels.to(DEVICE)

                outputs = model(spectrograms)
                loss = criterion(outputs, labels)

                val_loss += loss.item()
                _, predicted = outputs.max(1)
                val_total += labels.size(0)
                val_correct += predicted.eq(labels).sum().item()

        # Calculate validation metrics
        val_loss = val_loss / len(val_loader)
        val_acc = 100. * val_correct / val_total

        # Step the scheduler
        current_lr = optimizer.param_groups[0]['lr']
        scheduler.step()

        # Print epoch summary
        print(f'\n{"="*60}')
        print(f'Epoch {epoch+1}/{NUM_EPOCHS} Summary:')
        print(f'Learning Rate: {current_lr:.6f}')
        print(f'Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.2f}%')
        print(f'Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.2f}%')
        print(f'{"="*60}\n')

        # Save checkpoint
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            torch.save(model.state_dict(), 'best_model.pth')
            print(f'Saved new best model with validation accuracy: {val_acc:.2f}%\n')

    print("Training complete!")

    torch.save(model.state_dict(), 'stage3_severity_classifier.pth')

    checkpoint = torch.load('stage3_severity_classifier.pth')
    model.load_state_dict(checkpoint['model_state_dict'])

    model.eval()
    test_preds = []
    test_labels = []

    with torch.no_grad():
        for specs, labels in val_loader:
            specs = specs.to(DEVICE)
            outputs = model(specs)
            _, predicted = outputs.max(1)
            test_preds.extend(predicted.cpu().numpy())
            test_labels.extend(labels.cpu().numpy())

    test_acc = 100. * sum(p == l for p, l in zip(test_preds, test_labels)) / len(test_labels)
    test_f1 = f1_score(test_labels, test_preds, average='weighted')

    print(f"\n📊 Test Results:")
    print(f"  Accuracy: {test_acc:.2f}%")
    print(f"  F1-Score: {test_f1:.4f}")

    print(f"\n🔢 Confusion Matrix:")
    cm = confusion_matrix(test_labels, test_preds)
    print("\n          Predicted →")
    print("Actual ↓  ", "  ".join([f"{LABEL_NAMES[i]:9s}" for i in range(3)]))
    for i, row in enumerate(cm):
        print(f"{LABEL_NAMES[i]:9s}", "  ".join([f"{val:9d}" for val in row]))

    print(f"\n📋 Per-Class Report:")
    print(classification_report(
        test_labels, test_preds,
        target_names=[LABEL_NAMES[i] for i in range(3)],
        digits=4
    ))

    print("\n" + "="*70)
    print("DONE!")
    print("="*70)

train_stage3_severity_classification()


Found 272 files in /workspace/task1/mel_spectrograms/phonationA
Found 272 files in /workspace/task1/mel_spectrograms/phonationE
Found 272 files in /workspace/task1/mel_spectrograms/phonationI
Found 272 files in /workspace/task1/mel_spectrograms/phonationO
Found 272 files in /workspace/task1/mel_spectrograms/phonationU
Found 272 files in /workspace/task1/mel_spectrograms/rhythmKA
Found 272 files in /workspace/task1/mel_spectrograms/rhythmPA
Found 272 files in /workspace/task1/mel_spectrograms/rhythmTA

=== Dataset Summary ===
Total samples: 2176
Class distribution:
  Class 1 (Severe dysarthria): 48 samples
  Class 2 (Moderate dysarthria): 208 samples
  Class 3 (Mild dysarthria): 456 samples
  Class 4 (ALS without dysarthria): 608 samples
  Class 5 (Healthy): 856 samples

=== VALIDATING ALL LABELS ===
Min label: 0
Max label: 2
Unique labels: [0, 1, 2]
Label distribution:
  Label 0: 48 samples
  Label 1: 208 samples
  Label 2: 456 samples
✓ All labels valid!



NameError: name 'BATCH_SIZE' is not defined

In [19]:
checkpoint = torch.load('stage3_severity_classifier.pth')
model.load_state_dict(checkpoint['model_state_dict'])

model.eval()
test_preds = []
test_labels = []

with torch.no_grad():
    for specs, labels in test_loader:
        specs = specs.to(DEVICE)
        outputs = model(specs)
        _, predicted = outputs.max(1)
        test_preds.extend(predicted.cpu().numpy())
        test_labels.extend(labels.cpu().numpy())

test_acc = 100. * sum(p == l for p, l in zip(test_preds, test_labels)) / len(test_labels)
test_f1 = f1_score(test_labels, test_preds, average='weighted')

print(f"\n📊 Test Results:")
print(f"  Accuracy: {test_acc:.2f}%")
print(f"  F1-Score: {test_f1:.4f}")

print(f"\n🔢 Confusion Matrix:")
cm = confusion_matrix(test_labels, test_preds)
print("\n          Predicted →")
print("Actual ↓  ", "  ".join([f"{LABEL_NAMES[i]:9s}" for i in range(3)]))
for i, row in enumerate(cm):
    print(f"{LABEL_NAMES[i]:9s}", "  ".join([f"{val:9d}" for val in row]))

print(f"\n📋 Per-Class Report:")
print(classification_report(
    test_labels, test_preds,
    target_names=[LABEL_NAMES[i] for i in range(3)],
    digits=4
))

print("\n" + "="*70)
print("DONE!")
print("="*70)

NameError: name 'model' is not defined

#Run Classifier

In [ ]:
def load_model(checkpoint_path, label_dim, device=None):
    """
    Load a trained AST model from a checkpoint file

    Parameters:
    -----------
    checkpoint_path : str
        Path to the saved model weights (.pth file)
    label_dim : int
        Number of output classes (2 for binary, 3 for severity)
    device : torch.device, optional
        Device to load model on. If None, uses GPU if available

    Returns:
    --------
    model : ASTModel
        Loaded model in evaluation mode
    """
    if device is None:
        device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

    # Create model with same architecture as training
    model = ASTModel(
        label_dim=label_dim,
        fstride=10,
        tstride=10,
        input_fdim=128,
        input_tdim=1024,
        imagenet_pretrain=False,  # Don't need pretrained weights when loading
        audioset_pretrain=False,
        model_size='base384'
    )

    # Load saved weights
    model.load_state_dict(torch.load(checkpoint_path, map_location=device))

    # Move to device and set to evaluation mode
    model = model.to(device)
    model.eval()

    return model

def predict_hierarchical_3stage(spectrogram):
    # Load all models
    als_model = load_model('stage1_als_detector.pth', label_dim=2)
    dysarthria_model = load_model('stage2_dysarthria_detector.pth', label_dim=2)
    severity_model = load_model('stage3_severity_classifier.pth', label_dim=3)

    with torch.no_grad():
        # Stage 1: ALS Detection
        als_output = als_model(spectrogram)
        als_prob = torch.softmax(als_output, dim=1)
        als_pred = torch.argmax(als_prob, dim=1)

        if als_pred == 0:
            return {
                'diagnosis': 'Healthy',
                'confidence': als_prob[0][0].item(),
                'stage': 1
            }

        # Stage 2: Dysarthria Detection
        dysarthria_output = dysarthria_model(spectrogram)
        dysarthria_prob = torch.softmax(dysarthria_output, dim=1)
        dysarthria_pred = torch.argmax(dysarthria_prob, dim=1)

        if dysarthria_pred == 0:
            return {
                'diagnosis': 'ALS without dysarthria',
                'confidence': dysarthria_prob[0][0].item(),
                'stage': 2
            }

        # Stage 3: Severity Classification
        severity_output = severity_model(spectrogram)
        severity_prob = torch.softmax(severity_output, dim=1)
        severity_pred = torch.argmax(severity_prob, dim=1)

        severity_labels = ['Mild', 'Moderate', 'Severe']
        return {
            'diagnosis': f'ALS with {severity_labels[severity_pred]} dysarthria',
            'confidence': severity_prob[0][severity_pred].item(),
            'stage': 3,
            'severity': severity_labels[severity_pred]
        }

predict_hierarchical_3stage

In [ ]:
!python train.py

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(12, 4))

# Plot loss
plt.subplot(1, 2, 1)
plt.plot(train_losses, label='Train Loss')
plt.plot(val_losses, label='Val Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Training and Validation Loss')
plt.legend()
plt.grid(True)

# Plot accuracy
plt.subplot(1, 2, 2)
plt.plot(train_accs, label='Train Acc')
plt.plot(val_accs, label='Val Acc')
plt.xlabel('Epoch')
plt.ylabel('Accuracy (%)')
plt.title('Training and Validation Accuracy')
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.savefig('training_curves.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
from google.colab import files

# Download the best model
files.download('best_model.pth')

# Download training curves
files.download('training_curves.png')